# Building multi-model AI chatbot using Gradio Interface, integrating Gemini and OpenRouter APIs, and managing secure user authentication.

In [ ]:
!pip install -q gradio google-generativeai openai

In [ ]:
from gradio import ChatInterface

In [ ]:
def sample_fn(prompt, history):
    messages = []
    for value in history:
        messages.append({
            "role": value["role"],
            "content": value["content"]
        })
    return f'The current prompt is {prompt}, history is {history}'

In [ ]:
demo= ChatInterface(fn=sample_fn, examples=['hello','hola','hi'], title="Echo Bot")
demo.launch(debug=True, share=True)

TASK--

In [ ]:
from openai import OpenAI
import google.generativeai as genai
import gradio as gr
from google.colab import userdata

In [ ]:
api_key=userdata.get('GEMINI_API')
key_api=userdata.get('api')

In [ ]:
if api_key:
    genai.configure(api_key=api_key)

In [ ]:
client = OpenAI(
     base_url='https://openrouter.ai/api/v1',
     api_key=key_api
 )

In [ ]:
system_prompt= 'You are a helpful assistant'

In [ ]:
def gemini_response(message):

    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_prompt
    )

    chat = model.start_chat(history=[])

    response = chat.send_message(message)

    return response.text

In [ ]:
def openrouter_response(message):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": message}
    ]
    response = client.chat.completions.create(
        model="openai/gpt-4.1-mini",
        messages=messages
        max_tokens=100
    )
    return response.choices[0].message.content

In [ ]:
def chatbot(message, model):

    if model == "GPT":
        return openrouter_response(message)

    elif model == "Gemini":
        return gemini_response(message)

    else:
        return "Invalid model selected."

In [ ]:
model_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value= "GPT")
message_output= gr.Textbox(label= "Output", lines= 7)

demo = gr.Interface(
    fn=chatbot,
    inputs= [model_input, model_selector],
    outputs= [message_output],
    title="AI Chatbot",
    description="Chat with GPT (OpenRouter) or Gemini."
)

demo.launch(debug=True, auth=("Gurdeep","005"))